Question 1: Image Preprocessing for Inference (PyTorch)
Problem: Write a function to load an image and preprocess it for inference.

In [1]:
import torch
from torchvision import transforms
from PIL import Image

def preprocess_image(image_path, device="cpu"):
    preprocess = transforms.Compose([
        transforms.Resize(256),                # Resize shortest side to 256
        transforms.CenterCrop(224),            # Crop to 224x224
        transforms.ToTensor(),                 # Convert to tensor [0,1]
        transforms.Normalize(                  # Normalize with ImageNet stats
            mean=[0.485, 0.456, 0.406], 
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    # Load image
    img = Image.open(image_path).convert("RGB")
    
    # Apply transforms
    img_tensor = preprocess(img)
    
    # Add batch dimension: (1, 3, 224, 224)
    img_tensor = img_tensor.unsqueeze(0).to(device)
    
    return img_tensor

Question 2: Predict on New Image with a Trained Model
Problem: Perform prediction and get the class label.

In [2]:
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input, decode_predictions
import numpy as np
import keras

model = keras.applications.MobileNet(
    input_shape=None,
    alpha=1.0,
    depth_multiplier=1,
    dropout=0.001,
    include_top=True,
    weights="imagenet",
    input_tensor=None,
    pooling=None,
    classes=1000,
    classifier_activation="softmax",
    name=None,
)
img_path = 'tiger.jpg'   # path to your image
img = image.load_img(img_path, target_size=(224, 224))  # resize to 224x224

# Convert to array
x = image.img_to_array(img)

# Add batch dimension (model expects batch of images)
x = np.expand_dims(x, axis=0)

# Preprocess for MobileNetV2
x = preprocess_input(x)
preds = model.predict(x)

print(np.argmax(preds))
decoded= decode_predictions(preds,top=1)[0]

label = decoded[0][1]
print('Predicted:',label)

1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
292
Predicted: tiger


Question 3: Build a CNN to classify CIFAR-10 images (PyTorch)
Problem: Create a CNN model that classifies images from the CIFAR-10 dataset with accuracy above 60%.

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms

# 1. Load CIFAR-10 dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))  # normalize to [-1,1]
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64,
                                          shuffle=True)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64,
                                         shuffle=False)

# 2. Build CNN
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 64),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(64, 10)
        )


    def forward(self, x):
        return self.model(x)

model = CNN()

# 3. Compile (loss + optimizer)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 4. Train
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(6):  # 6 epochs
    model.train()
    running_loss = 0.0
    correct, total = 0, 0

    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total
    print(f"Epoch {epoch+1}, Loss: {running_loss/len(trainloader):.4f}, Train Acc: {train_acc:.2f}%")

# 5. Evaluate
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for inputs, labels in testloader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test accuracy: {100 * correct / total:.2f}%")

Files already downloaded and verified
Files already downloaded and verified
Epoch 1, Loss: 1.7535, Train Acc: 34.35%
Epoch 2, Loss: 1.4885, Train Acc: 45.39%
Epoch 3, Loss: 1.3659, Train Acc: 50.23%
Epoch 4, Loss: 1.2639, Train Acc: 54.18%
Epoch 5, Loss: 1.1912, Train Acc: 57.41%
Epoch 6, Loss: 1.1318, Train Acc: 59.59%
Test accuracy: 65.81%


Question 4: Identify Overfitting from Training Logs and Solve It
Problem: You notice the training accuracy increases but validation accuracy stagnates. Modify the model using dropout and early stopping. (use mnist dataset)

In [4]:
from keras.datasets import mnist
from keras.layers import Dense, Activation, Dropout, Input, Conv2D,Flatten,MaxPool2D
import keras
from keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

(x_train,y_train),(x_test,y_test) = mnist.load_data()

x_train = x_train.reshape(x_train.shape[0],28,28,1)
x_test = x_test.reshape(x_test.shape[0],28,28,1)

x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0

y_train=to_categorical(y_train, num_classes=10)
y_test=to_categorical(y_test, num_classes=10)

model = keras.Sequential([
    Input(shape=(28,28,1)),
    Conv2D(28, kernel_size=3, activation='relu'),
    MaxPool2D(),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.2),
    Dense(10, activation='softmax')
])

model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['Accuracy'])

early_stopping = EarlyStopping(monitor='val_loss', patience=3)

history=model.fit(x=x_train, y= y_train, epochs=10)

model.evaluate(x_test,y_test)

Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 55s 22ms/step - Accuracy: 0.9559 - loss: 0.1492
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 38s 20ms/step - Accuracy: 0.9837 - loss: 0.0519
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 38s 20ms/step - Accuracy: 0.9901 - loss: 0.0330
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 38s 20ms/step - Accuracy: 0.9925 - loss: 0.0234
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 41s 20ms/step - Accuracy: 0.9941 - loss: 0.0179
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 38s 20ms/step - Accuracy: 0.9956 - loss: 0.0136
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 37s 20ms/step - Accuracy: 0.9965 - loss: 0.0111
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 37s 20ms/step - Accuracy: 0.9969 - loss: 0.0089
Epoch 9/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 38s 20ms/step - Accuracy: 0.9977 - loss: 0.0068
Epoch 10/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 37s 20ms/step - Accuracy: 0.9977 - loss: 0.0072
313/313 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - Accuracy: 0.9883 - loss: 0.0457


[0.04571530967950821, 0.9883000254631042]

Question 5: Transfer Learning with Pretrained VGG16 (Cats vs Dogs)
Problem: Use VGG16 for binary classification with fine-tuning

In [5]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Flatten, Dropout

base_model = VGG16(include_top=False, input_shape=(224, 224, 3), weights='imagenet')
for layer in base_model.layers:
    layer.trainable = False

x = base_model.output
x = Flatten()(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
